# 12 — Frozen confirmatory replication
This notebook executes `docs/confirmatory_protocol.md` on a disjoint asset panel. Do not change the assets, split, model settings, seeds, outcomes, or inference after viewing results. Any necessary deviation must be documented and the affected analysis relabeled exploratory.

In [1]:
import hashlib
import sys
from pathlib import Path
import numpy as np
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from innovcal.baselines import forecast_baselines
from innovcal.ca_rnn import CARNNConfig, TrainingConfig
from innovcal.data import clean_adjusted_prices, download_adjusted_prices
from innovcal.data.windows import Standardizer, chronological_split
from innovcal.evaluation import evaluate_samples, paired_seed_block_bootstrap
from innovcal.experiments import run_frequentist_seed_comparison
OUT = ROOT / 'results' / 'confirmatory'; OUT.mkdir(parents=True, exist_ok=True)
PROCESSED = ROOT / 'data' / 'processed'; PROCESSED.mkdir(parents=True, exist_ok=True)
protocol = ROOT / 'docs' / 'confirmatory_protocol.md'
protocol_hash = hashlib.sha256(protocol.read_bytes()).hexdigest()
print('Frozen protocol SHA-256:', protocol_hash)

Frozen protocol SHA-256: 20a0531461992b11d702d5b72b51c57a5e9b162e84dd5f6e6e34e5fadec570b0


## Acquire the frozen external panel
Only dimensions and date boundaries are shown before fitting. The cleaned panel is retained for auditability. The yfinance end date is exclusive.

In [2]:
ASSETS = ['MSFT', 'BAC', 'CVX', 'COST']
returns_path = PROCESSED / 'confirmatory_returns.csv'
if returns_path.exists():
    returns = pd.read_csv(returns_path, index_col=0, parse_dates=True)
else:
    raw = download_adjusted_prices(ASSETS, '2007-01-01', '2026-09-01')
    prices, returns = clean_adjusted_prices(raw)
    prices.to_csv(PROCESSED / 'confirmatory_prices.csv')
    returns.to_csv(returns_path)
returns = returns[ASSETS].dropna().sort_index()
assert returns.index.is_monotonic_increasing and not returns.index.has_duplicates
print(returns.shape, returns.index.min().date(), returns.index.max().date())

(4945, 4) 2007-01-04 2026-08-31


## Fit the two frozen neural specifications
This is the expensive cell: 10 seeds × 2 models = 20 fits. The temporal-penalty-removal ablation is excluded because the confirmatory primary contrast was frozen as CA-RNN versus RNN.

In [3]:
TRAINING_SEEDS = [611, 619, 631, 641, 647, 653, 661, 673, 683, 691]
training = TrainingConfig(epochs=200, batch_size=64, patience=25, seed=606, selection_metric='nll')
result = run_frequentist_seed_comparison(
    returns.to_numpy(), TRAINING_SEEDS, history=20,
    model_config=CARNNConfig(input_dim=4, hidden_dim=32),
    training_config=training, lambda_cal=100.0, lambda_seq=2000.0,
    projection_seed=606, forecast_seed=20_606, n_forecast_samples=500,
    models=('RNN', 'CA-RNN'),
)
assert np.isfinite(result.metrics.select_dtypes('number')).all().all()
display(result.metrics.groupby('model').agg(['mean', 'std']))

training_seed            best_epoch                rmse  \
                           mean        std       mean       std      mean   
model                                                                       
CA-RNN         651.0  26.566478       43.4  7.026932  0.869446   
RNN                       651.0  26.566478       29.8  5.633235  0.860916   

                            energy_score            coverage            ...  \
                        std         mean       std      mean       std  ...   
model                                                                   ...   
CA-RNN  0.001461     1.077784  0.002259  0.868327  0.005698  ...   
RNN                0.000601     1.067709  0.001078  0.870349  0.004019  ...   

                  pit_mean_absolute_autocorrelation           pit_mean_ks  \
                                               mean       std        mean   
model                                                                       
CA-RNN                          0.033779  0.002733    0.046151   
RNN                                        0.030079  0.001285    0.047900   

                            coordinate_pit_calibration_error            \
                        std                             mean       std   
model                                                                    
CA-RNN  0.003143                         0.000743  0.000188   
RNN                0.000933                         0.000901  0.000086   

                  coordinate_pit_mean_absolute_autocorrelation            \
                                                          mean       std   
model                                                                      
CA-RNN                                     0.032510  0.003612   
RNN                                                   0.026364  0.001611   

                  coordinate_pit_mean_ks            
                                    mean       std  
model                                               
CA-RNN               0.045892  0.005744  
RNN                             0.049593  0.002198  

[2 rows x 26 columns]

In [4]:
split = chronological_split(returns.to_numpy())
scaler = Standardizer.fit(split.train)
train, validation, test = map(scaler.transform, (split.train, split.validation, split.test))
baselines = forecast_baselines(train, validation, test, lag_candidates=(1, 5, 10, 20), n_samples=500, seed=20_606)
assert np.allclose(result.target, baselines.target, atol=1e-6)
baseline_metrics = pd.DataFrame([{'model': name, **evaluate_samples(result.target, samples, projections=result.projections)} for name, samples in baselines.samples.items()])
print('Selected VAR lag:', baselines.selected_lag)
print('GARCH convergence:', dict(zip(ASSETS, baselines.garch_converged)))
display(baseline_metrics)

Selected VAR lag: 1
GARCH convergence: {'MSFT': np.True_, 'BAC': np.True_, 'CVX': np.True_, 'COST': np.True_}


,model,rmse,energy_score,coverage,interval_width,interval_score,pit_calibration_error,pit_mean_absolute_autocorrelation,pit_mean_ks,coordinate_pit_calibration_error,coordinate_pit_mean_absolute_autocorrelation,coordinate_pit_mean_ks
0,Historical bootstrap,0.858240,1.069509,0.918605,2.782595,3.921384,0.000790,0.027726,0.044744,0.000793,0.022006,0.043827
1,VAR Gaussian,0.865754,1.102214,0.940091,3.256203,4.130710,0.004639,0.034039,0.092480,0.006580,0.030462,0.109727
2,VAR-GARCH bootstrap,0.864607,1.071332,0.892821,2.525109,3.752766,0.000491,0.032705,0.038460,0.000264,0.029642,0.031962


## Frozen inference
The first comparison is the sole primary contrast. All others are secondary context. Negative candidate-minus-comparator differences favor the candidate for loss metrics.

In [5]:
n_seeds = len(TRAINING_SEEDS)
combined = dict(result.forecast_samples)
combined.update({name: np.repeat(samples[None], n_seeds, axis=0) for name, samples in baselines.samples.items()})
comparisons = [('CA-RNN', 'RNN')]
comparisons += [(model, 'VAR-GARCH bootstrap') for model in ['RNN', 'CA-RNN']]
inference = paired_seed_block_bootstrap(result.target, combined, comparisons, result.projections, block_length=20, n_bootstrap=2000, confidence=0.95, seed=1208)
inference['analysis_status'] = np.where((inference.candidate == 'CA-RNN') & (inference.comparator == 'RNN'), 'primary', 'secondary')
inference['resolved'] = (inference.ci_lower > 0) | (inference.ci_upper < 0)
display(inference[inference.metric.isin(['energy_score', 'interval_score', 'coverage_error', 'pit_calibration_error'])][['analysis_status', 'candidate', 'comparator', 'metric', 'difference', 'ci_lower', 'ci_upper', 'resolved']])

,analysis_status,candidate,comparator,metric,difference,ci_lower,ci_upper,resolved
1,primary,CA-RNN,RNN,energy_score,0.010075,0.008472,0.011890,True
2,primary,CA-RNN,RNN,interval_score,0.023177,0.007734,0.039366,True
5,primary,CA-RNN,RNN,coverage_error,0.002022,-0.001188,0.005511,False
6,primary,CA-RNN,RNN,pit_calibration_error,-0.000047,-0.000163,-0.000012,True
9,secondary,RNN,VAR-GARCH bootstrap,energy_score,-0.003623,-0.005423,-0.001447,True
10,secondary,RNN,VAR-GARCH bootstrap,interval_score,0.023834,0.007825,0.050948,True
13,secondary,RNN,VAR-GARCH bootstrap,coverage_error,0.022472,0.017917,0.026340,True
14,secondary,RNN,VAR-GARCH bootstrap,pit_calibration_error,0.000291,0.000188,0.000377,True
17,secondary,CA-RNN,VAR-GARCH bootstrap,energy_score,0.006452,0.004251,0.009155,True
18,secondary,CA-RNN,VAR-GARCH bootstrap,interval_score,0.047011,0.030789,0.074281,True


In [6]:
result.metrics.to_csv(OUT / 'neural_seed_metrics.csv', index=False)
baseline_metrics.assign(selected_var_lag=baselines.selected_lag).to_csv(OUT / 'baseline_metrics.csv', index=False)
inference.to_csv(OUT / 'confirmatory_inference.csv', index=False)
pd.DataFrame({'asset': ASSETS, 'garch_optimizer_converged': baselines.garch_converged}).to_csv(OUT / 'garch_convergence.csv', index=False)
payload = {'target': result.target, 'projections': result.projections, 'training_seeds': np.asarray(TRAINING_SEEDS)}
for index, (name, samples) in enumerate(combined.items()): payload[f'samples_{index}'] = samples
np.savez_compressed(OUT / 'confirmatory_forecasts.npz', **payload)
(OUT / 'protocol_sha256.txt').write_text(protocol_hash + '\n')
pd.DataFrame({'model': list(combined), 'array_key': [f'samples_{i}' for i in range(len(combined))]}).to_csv(OUT / 'forecast_manifest.csv', index=False)
print('Saved all confirmatory outputs and the executed protocol hash.')

Saved all confirmatory outputs and the executed protocol hash.


## Mandatory reporting rule
Report all four primary metrics even if they disagree. A calibration improvement accompanied by worse proper scores is a tradeoff. Do not tune the method, replace assets, or redefine outcomes from this result; any subsequent modification starts a new exploratory development cycle.